## tl;dr
Official locally supplied data were profiled after a frozen header-only amendment. Full replay and 336 independent pandas checks pass. Baseline admission is false: actual human terms review and leakage-safe evaluation semantics remain pending. No model training or scoring is represented here.

## Context & Methods
Frozen protocol: workflow_intake_protocol.json. Source selection: PUBLIC_WORKFLOW_SELECTION.md.

### Key Assumptions
Keep all raw records unchanged. Split by normalized SMS group or whole incident, independently of labels. Do not infer chronology or field availability from enriched records. This is a read-only companion, not a downloader.

In [1]:
import json
from pathlib import Path
places = [Path.cwd(), *Path.cwd().parents]
here = next(p for root in places for p in (root, root / 'research/breakthrough') if (p / 'workflow_intake_protocol.json').is_file())
protocol = json.loads((here / 'workflow_intake_protocol.json').read_text())
print(json.dumps({'phase': protocol['phase'], 'split': protocol['split']['partitions'], 'network_policy': protocol['network_policy']}, indent=2))

{
  "phase": "official_archive_intake_and_quality_only_no_training_or_scoring",
  "split": {
    "train": [
      0,
      7000
    ],
    "development": [
      7000,
      8500
    ],
    "evaluation": [
      8500,
      10000
    ]
  },
  "network_policy": "current environment denies UCI browser access and terminal network; this runner is offline-only and cannot fetch, redirect, upload, or bypass restrictions"
}


## Data
Official dataset pages and DOI records are listed below. Archive hashes and local custody are recorded in the preserved run. Controller-supplied provenance is not independent authentication of the download. Raw messages and person identifiers are never displayed.

In [2]:
for lane, config in protocol['lanes'].items():
    print(lane, config['source_page'], config['doi'], config['documented_license'])

sms https://archive.ics.uci.edu/dataset/228/sms+spam+collection 10.24432/C5CC84 CC-BY-4.0
incidents https://archive.ics.uci.edu/dataset/498/incident+management+process+enriched+event+log 10.24432/C57S4H CC-BY-4.0


## Results
Inspect the fixed v2 receipt and verify its hash against the independent QA receipt. This notebook does not recreate the full raw-data audit; use the replay commands in WORKFLOW_INTAKE_V2_RESULTS.md for that. It does not create or repair a dataset.

In [3]:
import hashlib
run_directory = here / 'runs/workflow_intake_v2'
blob = (run_directory / 'result.json').read_bytes()
result = json.loads(blob)
qa = json.loads((run_directory / 'independent_qa.json').read_text())
assert hashlib.sha256(blob).hexdigest() == qa['checked_result_sha256']
assert result['baseline_admitted'] is False
assert result['training_runs'] == result['scoring_runs'] == 0
print(json.dumps({lane: {'status': info['status'], 'rows': info['profile']['rows'], 'split_sha256': info['split_sha256']} for lane, info in result['lanes'].items()}, indent=2))
print(json.dumps({'qa_checks': qa['checks_passed'], 'partitions': qa['partition_rows'], 'human_terms_review': result['human_terms_review']}, indent=2))
sms = result['lanes']['sms']['profile']
inc = result['lanes']['incidents']['profile']
print(json.dumps({'sms_cross_partition_templates': sms['digit_template_cross_partition_groups'], 'sms_affected_rows': sms['digit_template_cross_partition_rows'], 'incident_future_outcome_values': inc['future_outcome_values'], 'incident_regressions': inc['within_incident_file_order_regressions']}, indent=2))

{
  "incidents": {
    "status": "profiled_not_baseline_admitted",
    "rows": 141712,
    "split_sha256": "b624174e8f9dedc4115117ed4d1b5a17b9747509924c2fcb4fa40588059a014f"
  },
  "sms": {
    "status": "profiled_not_baseline_admitted",
    "rows": 5574,
    "split_sha256": "afe7bb594ec196af36150b92807e711d6728e73abf37d4f24905eb5f38a75ec8"
  }
}
{
  "qa_checks": 336,
  "partitions": {
    "sms": {
      "train": 3883,
      "evaluation": 847,
      "development": 844
    },
    "incidents": {
      "train": 98778,
      "development": 21582,
      "evaluation": 21352
    }
  },
  "human_terms_review": "pending_actual_human_review"
}
{
  "sms_cross_partition_templates": 4,
  "sms_affected_rows": 12,
  "incident_future_outcome_values": {
    "closed_at_after_record_update": 116726,
    "resolved_at_after_record_update": 87471,
    "update_before_opened": 5
  },
  "incident_regressions": {
    "timestamp": 1,
    "update_counter": 4
  }
}


## Takeaways
SMS template overlap (4 groups / 12 rows) blocks semantic-generalization scoring until a leakage-safe subset is frozen. Incident enrichment is not proof of event-time observability; preserve missingness and chronology anomalies. Actual human review of both source-page terms and SMS README is pending. Neither software checks nor this intake establish scientific novelty or product utility. See WORKFLOW_INTAKE_V2_RESULTS.md for denominators and remediation, and R8_NEXT_GATE_BRIEF.md for the separate scientific handoff.

Execution limitation: standard Python sequential code-cell replay only; no Jupyter kernel/nbclient execution is claimed.